# 03 — Entrenamiento del Modelo
**VeneCapital | MobileNetV2 Transfer Learning**

> ⚡ **Recomendado ejecutar en Google Colab con GPU**
>
> Runtime → Change runtime type → GPU

Este notebook:
- Carga los datos y construye los tf.data.Datasets
- Entrena el modelo (Fase 1: Feature Extraction)
- Opcionalmente aplica Fine-Tuning (Fase 2)
- Guarda el modelo y el historial de entrenamiento

In [ ]:
# Si estás en Google Colab, monta Drive primero:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Proyecto

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Carga del dataset ─────────────────────────────────────────────────────
from src.data_loader import get_datasets

train_ds, val_ds, test_ds, class_weights = get_datasets()
print(f'\n⚖️  Pesos de clase: {class_weights}')

In [ ]:
# ── Construcción del modelo ───────────────────────────────────────────────
from src.model import build_model, print_model_summary

model = build_model(fine_tune=False)
print_model_summary(model)

In [ ]:
# ── FASE 1: Feature Extraction ────────────────────────────────────────────
from src.train import get_callbacks

history = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=get_callbacks(),
    class_weight=class_weights,
    verbose=1
)

In [ ]:
# ── FASE 2 (Opcional): Fine-Tuning ────────────────────────────────────────
# Descomentar si quieres aplicar fine-tuning después del entrenamiento base

# from src.train import train_phase2
# history_ft = train_phase2(model, train_ds, val_ds, class_weights=class_weights, epochs=10)

print('Fine-tuning desactivado. Descomentar el bloque anterior para activarlo.')

In [ ]:
# ── Guardar modelo e historial ────────────────────────────────────────────
from src.train import save_training_history, save_model

history_data = save_training_history(history)
save_model(model)

print('\n✅ Modelo e historial guardados correctamente.')

In [ ]:
# ── Vista rápida de las curvas de aprendizaje ─────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1A1A2E')

for ax in axes:
    ax.set_facecolor('#1A1A2E')

axes[0].plot(history.history['loss'], label='Train Loss', color='#FF6584')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#FF6584', linestyle='--')
axes[0].set_title('Loss', color='#6C63FF', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].tick_params(colors='white')

axes[1].plot(history.history['auc'], label='Train AUC', color='#6C63FF')
axes[1].plot(history.history['val_auc'], label='Val AUC', color='#6C63FF', linestyle='--')
axes[1].set_title('AUC-ROC', color='#6C63FF', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].tick_params(colors='white')

plt.tight_layout()
plt.show()